## 기본 예시: 프롬프트 + 모델 + 출력 파서

가장 기본적이고 일반적인 사용 사례는 prompt 템플릿과 모델을 함께 연결하는 것입니다. 이것이 어떻게 작동하는지 보기 위해, 각 나라별 수도를 물어보는 Chain을 생성해 보겠습니다.


In [1]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

False

In [2]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH01-Basic")

LangChain/LangSmith API Key가 설정되지 않았습니다. 참고: https://wikidocs.net/250954


## 프롬프트 템플릿의 활용

`PromptTemplate`

- 사용자의 입력 변수를 사용하여 완전한 프롬프트 문자열을 만드는 데 사용되는 템플릿입니다
- 사용법
  - `template`: 템플릿 문자열입니다. 이 문자열 내에서 중괄호 `{}`는 변수를 나타냅니다.
  - `input_variables`: 중괄호 안에 들어갈 변수의 이름을 리스트로 정의합니다.

`input_variables`

- input_variables는 PromptTemplate에서 사용되는 변수의 이름을 정의하는 리스트입니다.

In [3]:
from langchain_teddynote.messages import stream_response  # 스트리밍 출력
from langchain_core.prompts import PromptTemplate

`from_template()` 메소드를 사용하여 PromptTemplate 객체 생성


In [4]:
# template 정의
template = "{country}의 수도는 어디인가요?"

# from_template 메소드를 이용하여 PromptTemplate 객체 생성
prompt_template = PromptTemplate.from_template(template)
prompt_template

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?')

In [5]:
# prompt 생성
prompt = prompt_template.format(country="대한민국")
prompt

'대한민국의 수도는 어디인가요?'

In [6]:
# prompt 생성
prompt = prompt_template.format(country="미국")
prompt

'미국의 수도는 어디인가요?'

In [9]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="gemma4:26b",
    temperature=0.1,
)

## Chain 생성

### LCEL(LangChain Expression Language)

![lcel.png](./images/lcel.png)

여기서 우리는 LCEL을 사용하여 다양한 구성 요소를 단일 체인으로 결합합니다

```
chain = prompt | model | output_parser
```

`|` 기호는 [unix 파이프 연산자](<https://en.wikipedia.org/wiki/Pipeline_(Unix)>)와 유사하며, 서로 다른 구성 요소를 연결하고 한 구성 요소의 출력을 다음 구성 요소의 입력으로 전달합니다.

이 체인에서 사용자 입력은 프롬프트 템플릿으로 전달되고, 그런 다음 프롬프트 템플릿 출력은 모델로 전달됩니다. 각 구성 요소를 개별적으로 살펴보면 무슨 일이 일어나고 있는지 이해할 수 있습니다.


In [10]:
# prompt 를 PromptTemplate 객체로 생성합니다.
prompt = PromptTemplate.from_template("{topic} 에 대해 쉽게 설명해주세요.")

model = ChatOllama (model="gemma4:26b", temperature=0.1)

chain = prompt | model

### invoke() 호출

- python 딕셔너리 형태로 입력값을 전달합니다.(키: 값)
- invoke() 함수 호출 시, 입력값을 전달합니다.

In [11]:
# input 딕셔너리에 주제를 '인공지능 모델의 학습 원리'으로 설정합니다.
input = {"topic": "인공지능 모델의 학습 원리"}

In [12]:
# prompt 객체와 model 객체를 파이프(|) 연산자로 연결하고 invoke 메서드를 사용하여 input을 전달합니다.
# 이를 통해 AI 모델이 생성한 메시지를 반환합니다.
chain.invoke(input)

AIMessage(content='인공지능(AI)이 학습하는 원리는 복잡한 수학 공식으로 가득 차 있지만, 핵심 원리는 우리가 **\'아이에게 사과와 배를 구별하는 법을 가르치는 과정\'**과 매우 비슷합니다.\n\n이 과정을 4단계로 나누어 아주 쉽게 설명해 드릴게요.\n\n---\n\n### 1. 데이터 입력 (공부할 재료 준비)\n먼저 아이(AI)에게 수만 장의 사과 사진과 배 사진을 보여줍니다. 이때 사진에는 사과의 특징(빨간색, 둥근 모양)과 배의 특징(노란색, 약간 울퉁불퉁한 모양)이 담겨 있습니다.\n\n*   **AI 용어:** 이를 **데이터(Data)** 또는 **학습 데이터(Training Data)**라고 부릅니다.\n\n### 2. 예측 (일단 찍어보기)\n처음에 AI는 아무것도 모르는 상태입니다. 그래서 사과 사진을 보여주면 무작정 "이건 배 같아요!"라고 찍습니다. 처음에는 당연히 틀릴 확률이 매우 높겠죠?\n\n*   **AI 용어:** AI가 내놓은 답을 **예측(Prediction)**이라고 합니다.\n\n### 3. 오차 계산 (틀린 정도 확인하기)\n아이가 "이건 배예요!"라고 했는데 정답이 사과였다면, 선생님(개발자/데이터)이 **"틀렸어! 이건 사과야. 너는 정답과 이만큼이나 차이가 나!"**라고 알려줍니다. \n\n이때 **\'정답과 AI가 예측한 값 사이의 거리(차이)\'**를 계산하는 것이 핵심입니다.\n\n*   **AI 용어:** 이 차이를 계산하는 공식을 **손실 함수(Loss Function)**라고 합니다. 이 값이 클수록 AI가 많이 틀렸다는 뜻입니다.\n\n### 4. 가중치 수정 (틀린 이유를 찾아 공부하기)\n이제 가장 중요한 단계입니다. AI는 왜 틀렸는지 분석합니다. \n*\'아, 내가 색깔을 중요하게 안 봤구나. 다음부터는 빨간색인지 확인을 더 열심히 해야지!\'* 라고 스스로 판단하며, 판단 기준을 수정합니다.\n\n이 과정을 반복하면 AI는 점점 정답에 가까운 판단 기준을 갖게 됩니다.\n\

아래는 스트리밍을 출력하는 예시 입니다.

In [13]:
# 스트리밍 출력을 위한 요청
answer = chain.stream(input)
# 스트리밍 출력
stream_response(answer)

인공지능(AI)이 학습하는 원리는 복잡한 수학 공식으로 가득 차 있지만, 핵심 개념은 우리가 **'공부를 해서 실력을 키우는 과정'**과 매우 비슷합니다.

이해를 돕기 위해 **"수학 문제를 푸는 학생"**에 비유해서 4단계로 설명해 드릴게요.

---

### 1. 데이터 (Data): "문제집과 정답지"
공부를 하려면 먼저 문제가 있어야겠죠? 인공지능에게 데이터는 **'학습 재료'**입니다.
*   개와 고양이를 구분하는 AI를 만든다면, 수만 장의 개 사진과 고양이 사진이 필요합니다.
*   이때 사진(문제)과 "이것은 개입니다"(정답)라는 정보가 세트로 필요합니다.

### 2. 예측 (Prediction): "일단 풀어보기 (찍기)"
처음 공부를 시작할 때 학생은 아무것도 모릅니다. 인공지능도 처음에는 아무런 지식이 없어서 무작위로 답을 내놓습니다.
*   AI가 사진을 보고 **"음... 이건 70% 확률로 고양이 같아!"**라고 예측을 합니다. 
*   이 과정을 전문 용어로 **'순전파(Forward Propagation)'**라고 합니다.

### 3. 오차 계산 (Loss Function): "채점하기"
예측을 했다면, 내가 쓴 답이 맞았는지 틀렸는지 확인해야 합니다.
*   AI가 "고양이"라고 했는데 실제 정답이 "개"였다면, **"틀렸어! 너는 이만큼 틀렸어"**라고 알려주는 과정입니다.
*   이 '틀린 정도(오차)'를 계산하는 것을 **'손실 함수(Loss Function)'**라고 합니다. 틀린 정도가 클수록 점수가 높게(나쁘게) 나옵니다.

### 4. 최적화 (Optimization/Backpropagation): "오답 노트 쓰고 공부하기"
이 단계가 **학습의 핵심**입니다. 틀렸다는 것을 알았으니, 왜 틀렸는지 분석하고 다음에는 안 틀리도록 머릿속의 지식을 수정해야 합니다.
*   **역전파(Backpropagation):** "아, 내가 귀 모양을 보고 고양이라고 착각했구나. 다음에는 귀 모양보다 코 모양을 더 중요하게

### 출력파서(Output Parser)


In [14]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

Chain 에 출력파서를 추가합니다.

In [15]:
# 프롬프트, 모델, 출력 파서를 연결하여 처리 체인을 구성합니다.
chain = prompt | model | output_parser

In [16]:
# chain 객체의 invoke 메서드를 사용하여 input을 전달합니다.
input = {"topic": "인공지능 모델의 학습 원리"}
chain.invoke(input)

'인공지능(AI)이 학습하는 원리는 복잡한 수학 공식으로 가득 차 있지만, 핵심 원리는 우리가 **\'양궁 선수가 과녁을 맞히기 위해 연습하는 과정\'**과 매우 비슷합니다.\n\n어려운 용어 대신, **\'농구 골대에 공을 넣는 연습\'**을 하는 아이의 모습에 비유해서 4단계로 설명해 드릴게요.\n\n---\n\n### 1. 데이터 입력 (공을 던지기 전의 준비)\n먼저 아이에게 수만 개의 농구공과 골대 위치를 보여줍니다. 인공지능에게 이것은 **\'데이터(Data)\'**입니다. \n*   **AI의 경우:** 사진, 글자, 숫자 같은 엄청난 양의 정보를 컴퓨터에게 입력합니다.\n\n### 2. 예측 (일단 던져보기)\n아이는 처음에는 힘 조절을 어떻게 해야 할지 모릅니다. 그래서 일단 감으로 공을 던져봅니다. "이 정도 힘이면 들어가겠지?" 하고 던지는 것이 인공지능의 **\'예측(Prediction)\'**입니다.\n*   **AI의 경우:** 입력된 데이터를 보고 "이 사진은 고양이일 확률이 60%야!"라고 자기 나름대로 답을 내놓습니다. (이때 AI 내부에는 \'가중치\'라는 조절 나사가 있는데, 처음에는 아무렇게나 설정되어 있습니다.)\n\n### 3. 손실 함수 (틀린 정도 확인하기)\n공을 던졌는데 골대보다 한참 멀리 날아갔습니다. 아이는 생각합니다. *"아, 너무 세게 던졌구나. 목표 지점에서 이만큼이나 벗어났네."* \n이렇게 **\'정답과 내가 내놓은 답이 얼마나 차이 나는지\'**를 계산하는 것을 **\'손실 함수(Loss Function)\'**라고 합니다.\n*   **AI의 경우:** 정답(고양이)과 AI의 답(강아지)을 비교해서 "얼마나 틀렸는지"를 숫자로 계산합니다. 틀린 정도가 클수록 \'손실(Loss)\'이 크다고 합니다.\n\n### 4. 최적화 및 역전파 (힘 조절 수정하기)\n이제 아이는 다음번에는 조금 더 살살 던지기로 결심합니다. "다음엔 팔을 조금 덜 굽히고 힘을 10%만 줄여보자!"라고 스스로를 수정합니다. 이

In [17]:
# 스트리밍 출력을 위한 요청
answer = chain.stream(input)
# 스트리밍 출력
stream_response(answer)

인공지능(AI)이 학습하는 원리는 복잡한 수학 공식으로 가득 차 있지만, 핵심 개념은 우리가 **'공부를 해서 실력을 키우는 과정'**과 매우 비슷합니다.

이해를 돕기 위해 **"양궁 선수가 과녁을 맞히는 연습을 하는 과정"**에 비유해서 설명해 드릴게요.

---

### 1. 핵심 요소 4가지 (비유)

먼저, AI 학습에 필요한 4가지 핵심 요소를 양궁 선수에 비유해 보겠습니다.

1.  **데이터 (Data) = 연습용 화살과 과녁**: 학습을 하기 위한 재료입니다. (예: 수만 장의 고양이 사진)
2.  **모델 (Model) = 양궁 선수의 뇌**: 화살을 어떻게 쏠지 결정하는 판단 기준입니다.
3.  **가중치 (Weight) = 근육의 힘과 각도**: 화살을 쏠 때 조절하는 미세한 힘의 강도나 각도입니다. (AI가 실제로 조정하는 값)
4.  **손실 함수 (Loss Function) = 과녁과의 거리**: 화살이 정중앙에서 얼마나 벗어났는지를 나타내는 '점수(오차)'입니다.

---

### 2. 학습의 4단계 과정 (사이클)

AI는 한 번에 똑똑해지지 않습니다. 아래의 4단계를 수만 번 반복하며 점점 정답에 가까워집니다.

#### **1단계: 예측하기 (Forward Propagation)**
선수가 일단 화살을 쏩니다. 처음에는 아무것도 모르기 때문에 아무렇게나 쏩니다. AI도 처음에는 데이터(사진)를 보고 "이건 강아지인가?" 하고 무작정 찍습니다.

#### **2단계: 틀린 정도 확인하기 (Loss Function)**
화살이 과녁의 정중앙에서 10cm 벗어났다면, "아, 10cm만큼 틀렸구나!"라고 확인합니다. AI도 정답(고양이)과 자신이 예측한 값(강아지) 사이의 차이를 계산합니다. 이 차이를 **'손실(Loss)'** 또는 **'오차'**라고 부릅니다.

#### **3단계: 원인 분석하기 (Backpropagation - 역전파)**
이 단계가 가장 중요합니다. 선수가 "왜 10cm나 벗어났지? 팔 각도가 너무 높았나? 

### 템플릿을 변경하여 적용

- 아래의 프롬프트 내용을 얼마든지 **변경** 하여 테스트 해볼 수 있습니다.
- `model_name` 역시 변경하여 테스트가 가능합니다.

In [19]:
template = """
당신은 영어를 가르치는 10년차 영어 선생님입니다. 주어진 상황에 맞는 영어 회화를 작성해 주세요.
양식은 [FORMAT]을 참고하여 작성해 주세요.

#상황:
{question}

#FORMAT:
- 영어 회화:
- 한글 해석:
"""

# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOllama 챗모델을 초기화합니다.
model = ChatOllama(model="gemma4:26b", temperature=0.1)

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

In [20]:
# 체인을 구성합니다.
chain = prompt | model | output_parser

In [21]:
# 완성된 Chain을 실행하여 답변을 얻습니다.
print(chain.invoke({"question": "저는 식당에 가서 음식을 주문하고 싶어요"}))

반갑습니다! 10년 차 영어 교사입니다. 식당에서 음식을 주문하는 상황은 영어 회화에서 가장 기본적이면서도 아주 중요한 부분이죠. 

너무 복잡한 문장보다는, 실제 현지 식당에서 가장 많이 쓰이는 **'자연스럽고 예의 바른 표현'** 위주로 대화문을 구성해 보았습니다. 이 대화문만 익혀두셔도 당황하지 않고 주문하실 수 있을 거예요.

---

- **영어 회화:**

**Server:** Hello! Are you ready to order?
**Customer:** Yes, I think so. I'd like to have the cheeseburger, please.
**Server:** Great. Would you like anything to drink with that?
**Customer:** I'll have an iced tea, please.
**Server:** Sure. Anything else for you?
**Customer:** No, that's all for now. Oh, and could I get some extra napkins?
**Server:** Of course. I'll bring them right away.

- **한글 해석:**

**점원:** 안녕하세요! 주문하시겠어요?
**손님:** 네, 준비됐어요. 치즈버거 하나 주세요.
**점원:** 좋습니다. 음료도 같이 드릴까요?
**손님:** 아이스티 한 잔 주세요.
**점원:** 네. 더 필요하신 건 없으신가요?
**손님:** 아니요, 지금은 그게 전부예요. 아, 그리고 냅킨 좀 더 주실 수 있나요?
**점원:** 물론이죠. 바로 가져다드릴게요.

---

**💡 선생님의 한 끗 차이 Tip!**

1.  **"I'd like to have ~"**: "I want ~"라고 해도 뜻은 통하지만, 식당에서는 "I'd like to have (메뉴 이름), please"라고 말하는 것이 훨씬 정중하고 세련된 표현입니다.
2.  **"That's all for 

In [22]:
# 완성된 Chain을 실행하여 답변을 얻습니다.
# 스트리밍 출력을 위한 요청
answer = chain.stream({"question": "저는 식당에 가서 음식을 주문하고 싶어요"})
# 스트리밍 출력
stream_response(answer)

반갑습니다! 10년 차 영어 교사입니다. 식당에서 음식을 주문하는 상황은 영어 회화에서 가장 기본적이면서도 실전 활용도가 매우 높은 상황이에요. 

단순히 "I want..."라고 하기보다는, 조금 더 정중하고 자연스러운 표현들을 사용하여 구성해 보았습니다. 이 대화문을 소리 내어 읽으며 연습해 보세요!

---

- **영어 회화:**

**Server:** Hello! Welcome to The Bistro. How many people are in your party?
**Customer:** Hi. A table for two, please.
**Server:** Certainly. Please follow me. Here are your menus. Can I start you off with something to drink?
**Customer:** I'll have an iced tea, please. And just water for my friend.
**Server:** Sure thing. Are you ready to order your main course, or do you need a few more minutes?
**Customer:** I think we're ready. What do you recommend here?
**Server:** Our Truffle Pasta is highly recommended. It's a customer favorite.
**Customer:** That sounds great. I'll have the Truffle Pasta, and my friend will have the Grilled Salmon.
**Server:** Excellent choices. Any dietary restrictions or allergies we should know about?
**Customer:** Actually, could I have the pasta without mushrooms? I'

In [23]:
# 이번에는 question 을 '미국에서 피자 주문'으로 설정하여 실행합니다.
# 스트리밍 출력을 위한 요청
answer = chain.stream({"question": "미국에서 피자 주문"})
# 스트리밍 출력
stream_response(answer)

반갑습니다! 10년 차 영어 교사입니다. 미국에서 피자를 주문할 때는 단순히 메뉴 이름만 말하기보다, **사이즈, 토핑 추가, 배달/포장 여부**를 명확히 전달하는 것이 중요합니다. 

실제 미국 현지 피자 가게에서 바로 사용할 수 있는 자연스러운 대화문을 준비했습니다.

---

- **영어 회화:**

**Employee:** Thanks for calling Pizza Palace! How can I help you today?
**Customer:** Hi, I'd like to place an order for delivery, please.
**Employee:** Sure thing. What can I get for you?
**Customer:** Can I get a large pepperoni pizza with extra cheese?
**Employee:** You got it. Would you like any sides or drinks with that?
**Customer:** Hmm, let me add an order of garlic knots and a Coke.
**Employee:** Okay. Anything else?
**Customer:** That's all for now.
**Employee:** Great. Can I have your address and phone number, please?
**Customer:** My address is 123 Maple Street, and my number is 555-0199.
**Employee:** Perfect. Your total is $28.50. It should be there in about 30 to 40 minutes.
**Customer:** Sounds good. Thank you!

---

- **한글 해석:**

**직원:** Pizza Palace입니다! 무엇을 도와드릴까요?
**손님:** 안녕하세요, 배달 주문 좀 하려고 하는데요.
